In [ ]:
# ── Mount Google Drive (all outputs persist here) ─────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ── Project root on Google Drive ──────────────────────
ROOT = Path('/content/drive/MyDrive/MM_HSD')
DIRS = {
    'raw':         ROOT / 'data/raw',
    'audio_wav':   ROOT / 'data/audio',
    'features':    ROOT / 'data/features',
    'audio_feat':  ROOT / 'data/features/audio',
    'text_feat':   ROOT / 'data/features/text',
    'transcripts': ROOT / 'data/transcripts',
    'results':     ROOT / 'results',
    'configs':     ROOT / 'configs',
    'code':        ROOT / 'code',
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print('✅ All directories created:')
for k, v in DIRS.items():
    print(f'  {k}: {v}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All directories created:
  raw: /content/drive/MyDrive/MM_HSD/data/raw
  audio_wav: /content/drive/MyDrive/MM_HSD/data/audio
  features: /content/drive/MyDrive/MM_HSD/data/features
  audio_feat: /content/drive/MyDrive/MM_HSD/data/features/audio
  text_feat: /content/drive/MyDrive/MM_HSD/data/features/text
  transcripts: /content/drive/MyDrive/MM_HSD/data/transcripts
  results: /content/drive/MyDrive/MM_HSD/results
  configs: /content/drive/MyDrive/MM_HSD/configs
  code: /content/drive/MyDrive/MM_HSD/code


In [ ]:
# ── Install all required packages ─────────────────────
# Colab already has: torch, ffmpeg, tqdm, pandas, numpy

print('Installing openai-whisper...')
!pip install -q openai-whisper

print('Installing detoxify...')
!pip install -q detoxify

print('Installing transformers + audio libs...')
!pip install -q transformers==4.40.0 accelerate soundfile librosa

print('Installing configargparse (needed by mm-hsd trainer)...')
!pip install -q configargparse wandb

print('\n✅ All packages installed. Restarting runtime is NOT needed.')

Installing openai-whisper...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 49.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Installing detoxify...
Installing transformers + audio libs...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
Installing configargparse (needed by mm-hsd trainer)...

✅ All packages instal

In [ ]:
# ── Clone MM-HSD official code ────────────────────────
CODE_DIR = DIRS['code']
REPO_DIR = CODE_DIR / 'mm-hsd'

if not REPO_DIR.exists():
    !git clone https://github.com/idiap/mm-hsd.git {REPO_DIR}
    print('✅ MM-HSD repo cloned')
else:
    print('✅ MM-HSD repo already exists, pulling latest...')
    !git -C {REPO_DIR} pull

# Install mm-hsd package (CPU installs torch; Colab already has GPU torch)
import sys
!{sys.executable} -m pip install -q -e "{REPO_DIR}" --no-deps
print('✅ mm_hsd package installed')

✅ MM-HSD repo already exists, pulling latest...
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Package 'mm-hsd' requires a different Python: 3.12.12 not in '<3.10,>=3.9'
✅ mm_hsd package installed


In [ ]:
# ── Download HateMM annotation CSV (small, always download) ──
import urllib.request
import os

CSV_URL  = 'https://zenodo.org/records/7799469/files/HateMM_annotation.csv?download=1'
CSV_PATH = DIRS['raw'] / 'HateMM_annotation.csv'

if not CSV_PATH.exists():
    print('Downloading annotation CSV...')
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)
    print(f'✅ Saved: {CSV_PATH}')
else:
    print(f'✅ Annotation CSV already downloaded: {CSV_PATH}')

import pandas as pd
df = pd.read_csv(CSV_PATH)
print(f'\nDataset summary:')
print(f'  Total videos : {len(df)}')
if 'label' in df.columns:
    print(df['label'].value_counts())
print(df.head())

✅ Annotation CSV already downloaded: /content/drive/MyDrive/MM_HSD/data/raw/HateMM_annotation.csv

Dataset summary:
  Total videos : 1083
label
Non Hate    652
Hate        431
Name: count, dtype: int64
        video_file_name     label  \
0      hate_video_1.mp4      Hate   
1      hate_video_2.mp4      Hate   
2  non_hate_video_1.mp4  Non Hate   
3      hate_video_3.mp4      Hate   
4  non_hate_video_2.mp4  Non Hate   

                                        hate_snippet  target  
0                         [['00:00:34', '00:01:34']]  Blacks  
1                         [['00:00:06', '00:02:06']]  Blacks  
2                                                NaN  Others  
3  [['00:00:03', '00:01:40'], ['00:01:41', '00:03...  Blacks  
4                                                NaN  Blacks  


In [ ]:
# ── Download hate + non-hate videos (6.3 GB total) ───────
# This uses wget for resumable downloads with progress

HATE_ZIP     = DIRS['raw'] / 'hate_videos.zip'
NONHATE_ZIP  = DIRS['raw'] / 'non_hate_videos.zip'
HATE_DIR     = DIRS['raw'] / 'hate'
NONHATE_DIR  = DIRS['raw'] / 'non_hate'

HATE_DIR.mkdir(exist_ok=True)
NONHATE_DIR.mkdir(exist_ok=True)

# Download hate videos
if not HATE_ZIP.exists():
    print('⬇️  Downloading hate_videos.zip (~2.8 GB)...')
    !wget -c 'https://zenodo.org/records/7799469/files/hate_videos.zip?download=1' \
         -O {HATE_ZIP} --show-progress
else:
    print(f'✅ hate_videos.zip already downloaded')

# Download non-hate videos
if not NONHATE_ZIP.exists():
    print('⬇️  Downloading non_hate_videos.zip (~3.5 GB)...')
    !wget -c 'https://zenodo.org/records/7799469/files/non_hate_videos.zip?download=1' \
         -O {NONHATE_ZIP} --show-progress
else:
    print(f'✅ non_hate_videos.zip already downloaded')

✅ hate_videos.zip already downloaded
✅ non_hate_videos.zip already downloaded


In [ ]:
# ── Extract videos ────────────────────────────────────
import zipfile

def extract_if_needed(zip_path, out_dir, label):
    existing = list(out_dir.rglob('*.mp4'))
    if len(existing) > 10:
        print(f'✅ {label}: Already extracted ({len(existing)} files)')
        return
    print(f'📦 Extracting {zip_path.name} → {out_dir} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    extracted = list(out_dir.rglob('*.mp4'))
    print(f'✅ Extracted {len(extracted)} {label} videos')

extract_if_needed(HATE_ZIP,    HATE_DIR,    'hate')
extract_if_needed(NONHATE_ZIP, NONHATE_DIR, 'non_hate')

hate_count    = len(list(HATE_DIR.rglob('*.mp4')))
nonhate_count = len(list(NONHATE_DIR.rglob('*.mp4')))
print(f'\nFinal count — hate: {hate_count} | non-hate: {nonhate_count}')

✅ hate: Already extracted (431 files)
✅ non_hate: Already extracted (652 files)

Final count — hate: 431 | non-hate: 652


In [ ]:
# ── FFmpeg: extract mono 16kHz WAV from each video ───
# FFmpeg is pre-installed in Colab. This is fast (~15 min for all 1083 videos)

import subprocess
from tqdm.notebook import tqdm

def extract_wav(video_path: Path, out_dir: Path) -> bool:
    """Extract mono 16kHz PCM WAV. Returns True if done, False on error."""
    out_wav = out_dir / (video_path.stem + '.wav')
    if out_wav.exists() and out_wav.stat().st_size > 1000:
        return True  # already done
    cmd = [
        'ffmpeg', '-i', str(video_path),
        '-ac', '1',           # mono
        '-ar', '16000',       # 16 kHz (required by wav2vec2 + Whisper)
        '-sample_fmt', 's16', # PCM 16-bit
        '-vn',                # no video output
        str(out_wav), '-y', '-loglevel', 'error'
    ]
    result = subprocess.run(cmd, capture_output=True)
    return result.returncode == 0

errors = []
for label, vid_dir in [('hate', HATE_DIR), ('non_hate', NONHATE_DIR)]:
    out_dir = DIRS['audio_wav'] / label
    out_dir.mkdir(parents=True, exist_ok=True)
    videos = sorted(vid_dir.rglob('*.mp4'))
    print(f'\nExtracting audio from {len(videos)} {label} videos...')
    for v in tqdm(videos, desc=label):
        if not extract_wav(v, out_dir):
            errors.append(v.name)

hate_wavs    = len(list((DIRS['audio_wav']/'hate').rglob('*.wav')))
nonhate_wavs = len(list((DIRS['audio_wav']/'non_hate').rglob('*.wav')))
print(f'\n✅ WAVs ready — hate: {hate_wavs} | non-hate: {nonhate_wavs}')
if errors:
    print(f'⚠️  {len(errors)} failed: {errors[:5]}')


Extracting audio from 431 hate videos...


hate:   0%|          | 0/431 [00:00<?, ?it/s]


Extracting audio from 652 non_hate videos...


non_hate:   0%|          | 0/652 [00:00<?, ?it/s]


✅ WAVs ready — hate: 429 | non-hate: 639
⚠️  15 failed: ['hate_video_108.mp4', 'hate_video_17.mp4', 'non_hate_video_132.mp4', 'non_hate_video_2.mp4', 'non_hate_video_218.mp4']


In [ ]:
# ── Extract 1024-dim wav2vec2 features ───────────────
# Model: jonatasgrosman/wav2vec2-large-xlsr-53-english
# Output format: CSV with columns [file_name, feature_0, ..., feature_1023]

import csv, torch, numpy as np, soundfile as sf
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tqdm.notebook import tqdm

MODEL_ID = 'jonatasgrosman/wav2vec2-large-xlsr-53-english'
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Loading wav2vec2 on {device} (~1.2 GB download on first run)...')
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
w2v_model = Wav2Vec2Model.from_pretrained(MODEL_ID).to(device)
w2v_model.eval()
print('✅ wav2vec2 loaded')

HEADER = ['file_name'] + [f'feature_{i}' for i in range(1024)]
MAX_AUDIO_SEC = 90  # skip files longer than 10 min to avoid OOM

def extract_w2v_features(wav_path: Path) -> np.ndarray:
    """Returns 1024-dim mean-pooled last hidden state."""
    audio, sr = sf.read(str(wav_path))
    assert sr == 16000, f'Expected 16kHz, got {sr}'
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # ensure mono
    # Cap at MAX_AUDIO_SEC to avoid OOM on T4 (16 GB VRAM)
    audio = audio[:MAX_AUDIO_SEC * 16000]
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
    with torch.no_grad():
        outputs = w2v_model(inputs.input_values.to(device))
    return outputs.last_hidden_state.squeeze(0).mean(0).cpu().numpy()

for label in ['hate', 'non_hate']:
    out_csv  = DIRS['audio_feat'] / f'{label}_features.csv'
    wav_dir  = DIRS['audio_wav'] / label
    wavs     = sorted(wav_dir.rglob('*.wav'))

    # Load already-done file names for resumption
    done = set()
    if out_csv.exists():
        with open(out_csv) as f:
            done = {row[0] for row in csv.reader(f) if row}
        done.discard('file_name')

    todo = [w for w in wavs if w.name not in done]
    print(f'\n{label}: {len(done)} done, {len(todo)} remaining')
    if not todo:
        print('  ✅ All features already extracted!')
        continue

    write_header = not out_csv.exists() or len(done) == 0
    with open(out_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(HEADER)
        for wav in tqdm(todo, desc=f'wav2vec2 [{label}]'):
            try:
                feats = extract_w2v_features(wav)
                writer.writerow([wav.name] + feats.tolist())
                f.flush()  # flush every file so progress is saved on crash
            except Exception as e:
                print(f'  ⚠️  SKIP {wav.name}: {e}')

    count = len(list(open(out_csv))) - 1
    print(f'  ✅ {out_csv.name}: {count} rows saved')

Loading wav2vec2 on cuda (~1.2 GB download on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ wav2vec2 loaded

hate: 429 done, 0 remaining
  ✅ All features already extracted!

non_hate: 639 done, 0 remaining
  ✅ All features already extracted!


In [ ]:
# ── Transcribe WAVs using Whisper small ──────────────
# Output: transcripts.json  {video_name: "transcript text"}
# Whisper small is ~460 MB, runs well on T4

import json, whisper
from tqdm.notebook import tqdm

TRANSCRIPT_FILE = DIRS['transcripts'] / 'transcripts.json'

# Load existing transcripts for resumption
transcripts = {}
if TRANSCRIPT_FILE.exists():
    with open(TRANSCRIPT_FILE) as f:
        transcripts = json.load(f)
    print(f'Loaded {len(transcripts)} existing transcripts (resuming)')

print('Loading Whisper small model (~460 MB)...')
whisper_model = whisper.load_model('small', device='cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Whisper loaded')

for label in ['hate', 'non_hate']:
    wav_dir = DIRS['audio_wav'] / label
    wavs    = sorted(wav_dir.rglob('*.wav'))
    todo    = [w for w in wavs if w.stem not in transcripts]
    print(f'\n{label}: {len(wavs)-len(todo)} done, {len(todo)} remaining')

    for wav in tqdm(todo, desc=f'Whisper [{label}]'):
        try:
            result = whisper_model.transcribe(
                str(wav),
                language='en',
                fp16=torch.cuda.is_available(),  # fp16 on GPU for speed
                verbose=False
            )
            # ⚠️ DO NOT remove stopwords — paper shows it hurts M-F1
            transcripts[wav.stem] = result['text'].strip()
        except Exception as e:
            print(f'  ⚠️  SKIP {wav.name}: {e}')
            transcripts[wav.stem] = ''  # empty string fallback

        # Save every 10 files to survive Colab disconnections
        if len(transcripts) % 10 == 0:
            with open(TRANSCRIPT_FILE, 'w') as f:
                json.dump(transcripts, f, indent=2)

# Final save
with open(TRANSCRIPT_FILE, 'w') as f:
    json.dump(transcripts, f, indent=2)

empty = [k for k, v in transcripts.items() if not v.strip()]
print(f'\n✅ Total transcripts: {len(transcripts)}')
print(f'  Empty transcripts: {len(empty)} (will be encoded as empty string)')
print(f'  Saved → {TRANSCRIPT_FILE}')

Loaded 1010 existing transcripts (resuming)
Loading Whisper small model (~460 MB)...


100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 71.6MiB/s]


✅ Whisper loaded

hate: 429 done, 0 remaining


Whisper [hate]: 0it [00:00, ?it/s]


non_hate: 581 done, 58 remaining


Whisper [non_hate]:   0%|          | 0/58 [00:00<?, ?it/s]


 44%|████▍     | 11777/26777 [00:04<00:06, 2492.90frames/s]

100%|██████████| 2262/2262 [00:00<00:00, 4912.12frames/s]

100%|██████████| 20798/20798 [00:14<00:00, 1445.39frames/s]

100%|██████████| 23273/23273 [00:19<00:00, 1165.02frames/s]

100%|██████████| 12450/12450 [00:11<00:00, 1110.87frames/s]

 98%|█████████▊| 14228/14555 [00:28<00:00, 492.70frames/s] 

 22%|██▏       | 2600/11605 [00:02<00:08, 1021.51frames/s]

100%|██████████| 6560/6560 [00:08<00:00, 744.40frames/s]

100%|██████████| 517/517 [00:19<00:00, 26.44frames/s]

100%|██████████| 6181/6181 [00:03<00:00, 2037.19frames/s]

100%|██████████| 11536/11536 [00:06<00:00, 1782.28frames/s]

 77%|███████▋  | 20318/26318 [00:07<00:02, 2657.21frames/s]

100%|██████████| 8894/8894 [00:00<00:00, 22164.83frames/s]

100%|██████████| 10410/10410 [00:04<00:00, 2251.62frames/s]

100%|██████████| 6471/6471 [00:03<00:00, 1668.33frames/s]

100%|██████████| 3822/3822 [00:01<00:00, 2531.58frames/s]

100%|██████████| 4207/4207 [00:01<00:00, 2


✅ Total transcripts: 1068
  Empty transcripts: 43 (will be encoded as empty string)
  Saved → /content/drive/MyDrive/MM_HSD/data/transcripts/transcripts.json


In [ ]:
# ── Encode transcripts → 768-dim Detoxify embeddings ──
# Model: Detoxify 'original' (RoBERTa-based)
# Output format: JSON {video_name: [768 floats]}

import json, torch
from detoxify import Detoxify
from tqdm.notebook import tqdm

TRANSCRIPT_FILE = DIRS['transcripts'] / 'transcripts.json'
OUT_FILE        = DIRS['text_feat'] / 'embeddings_detoxify.json'

with open(TRANSCRIPT_FILE) as f:
    transcripts = json.load(f)

print(f'Loaded {len(transcripts)} transcripts')

# Load existing embeddings for resumption
embeddings = {}
if OUT_FILE.exists():
    with open(OUT_FILE) as f:
        embeddings = json.load(f)
    print(f'Resuming from {len(embeddings)} existing embeddings')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading Detoxify (RoBERTa) on {device}...')
dtx_obj   = Detoxify('original', device=device)
tokenizer = dtx_obj.tokenizer
bert_mdl  = dtx_obj.model
bert_mdl.eval()
print('✅ Detoxify loaded')

def encode_text(text: str) -> list:
    """Returns 768-dim [CLS] last-layer embedding."""
    if not text or not text.strip():
        text = '[empty]'  # graceful fallback
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        out = bert_mdl(**inputs, output_hidden_states=True)
    cls_emb = out.hidden_states[-1][:, 0, :].squeeze(0)
    return cls_emb.cpu().numpy().tolist()

todo = {k: v for k, v in transcripts.items() if k not in embeddings}
print(f'\nEncoding {len(todo)} transcripts...')

for i, (video_name, transcript) in enumerate(tqdm(todo.items())):
    embeddings[video_name] = encode_text(transcript)
    if i % 50 == 0:
        with open(OUT_FILE, 'w') as f:
            json.dump(embeddings, f)

with open(OUT_FILE, 'w') as f:
    json.dump(embeddings, f)

# Verify
sample_key = next(iter(embeddings))
print(f'\n✅ Text embeddings saved → {OUT_FILE}')
print(f'   Total: {len(embeddings)} videos')
print(f'   Dim:   {len(embeddings[sample_key])} (expected 768)')

Loaded 1068 transcripts
Loading Detoxify (RoBERTa) on cuda...
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:14<00:00, 29.8MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Detoxify loaded

Encoding 1068 transcripts...


  0%|          | 0/1068 [00:00<?, ?it/s]


✅ Text embeddings saved → /content/drive/MyDrive/MM_HSD/data/features/text/embeddings_detoxify.json
   Total: 1068 videos
   Dim:   768 (expected 768)


In [ ]:
# ── Sanity check all features before training ─────────
import csv, json

print('=' * 55)
print('    FEATURE VERIFICATION')
print('=' * 55)

all_ok = True

# 1. Audio features
for label in ['hate', 'non_hate']:
    csv_path = DIRS['audio_feat'] / f'{label}_features.csv'
    if not csv_path.exists():
        print(f'❌ MISSING: {csv_path}'); all_ok = False; continue
    with open(csv_path) as f:
        rows = list(csv.reader(f))
    n_rows = len(rows) - 1
    n_cols = len(rows[0]) - 1 if rows else 0
    status = '✅' if n_cols == 1024 else '❌'
    print(f'{status} Audio [{label}]: {n_rows} videos, {n_cols} features (expected 1024)')
    if n_cols != 1024: all_ok = False

# 2. Text embeddings
text_path = DIRS['text_feat'] / 'embeddings_detoxify.json'
if not text_path.exists():
    print(f'❌ MISSING: {text_path}'); all_ok = False
else:
    with open(text_path) as f:
        text_embs = json.load(f)
    sample_dim = len(next(iter(text_embs.values())))
    status = '✅' if sample_dim == 768 else '❌'
    print(f'{status} Text embeddings: {len(text_embs)} videos, dim={sample_dim} (expected 768)')
    if sample_dim != 768: all_ok = False

# 3. Cross-check: audio vs text coverage
with open(DIRS['audio_feat'] / 'hate_features.csv') as f:
    hate_audio = {row[0].replace('.wav','') for row in csv.reader(f) if row and row[0] != 'file_name'}
with open(DIRS['audio_feat'] / 'non_hate_features.csv') as f:
    nonhate_audio = {row[0].replace('.wav','') for row in csv.reader(f) if row and row[0] != 'file_name'}

all_audio = hate_audio | nonhate_audio
text_only = set(text_embs.keys()) - all_audio
audio_only = all_audio - set(text_embs.keys())

print(f'\n📊 Coverage:')
print(f'   Audio features : {len(all_audio)} videos')
print(f'   Text embeddings: {len(text_embs)} videos')
print(f'   In text but not audio: {len(text_only)}')
print(f'   In audio but not text: {len(audio_only)}')

print('\n' + ('=' * 55))
if all_ok:
    print('✅ ALL CHECKS PASSED — Ready to train!')
else:
    print('❌ Some checks failed — review above')

    FEATURE VERIFICATION
✅ Audio [hate]: 429 videos, 1024 features (expected 1024)
✅ Audio [non_hate]: 639 videos, 1024 features (expected 1024)
✅ Text embeddings: 1068 videos, dim=768 (expected 768)

📊 Coverage:
   Audio features : 1068 videos
   Text embeddings: 1068 videos
   In text but not audio: 0
   In audio but not text: 0

✅ ALL CHECKS PASSED — Ready to train!


In [ ]:
# ── Write config YAML for text + audio training ───────
import yaml

RESULTS_DIR = DIRS['results'] / 'text_audio'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = DIRS['configs'] / 'text_audio_only.yml'

config = {
    # ── Run info
    'run': 'text_audio_T4',

    # ── Training hyperparameters
    'batch_size': 8,
    'epochs': 40,
    'lr': 1e-5,
    'dropout': 0.4,
    'loss': 'CE',              # CrossEntropy
    'patience': 10,            # early stopping

    # ── Regularization
    'lambda_l1': 1e-5,
    'lambda_l2': 1e-4,

    # ── Modalities: text + audio ONLY
    'load_modalities': ['text', 'audio'],
    'include_modalities': ['text', 'audio'],
    'fusion': 'concat',        # concatenate encoded text + audio
    'audio_mode': 'wav2vec',

    # ── Input dimensions
    'input_size_text':  768,   # Detoxify/RoBERTa
    'input_size_audio': 1024,  # wav2vec2-large-xlsr-53
    'input_size_ocr':   768,   # placeholder (not used)
    'input_size_video': 768,   # placeholder (not used)

    # ── Hidden layer sizes
    'hidden_size_1_general': 128,
    'hidden_size_2_general': 128,
    'hidden_size_2_audio':   80,
    'output_size': 64,

    # ── Cross-validation
    'train_split_mode': 'automatic',
    'cross_fold': True,
    'folds': 5,

    # ── Data paths
    'dir_frames': None,
    'dir_audio':  str(DIRS['audio_feat']),
    'dir_text':   str(DIRS['text_feat'] / 'embeddings_detoxify.json'),
    'dir_ocr':    None,

    # ── Output
    'out_dir':  str(RESULTS_DIR),
    'log_dir':  str(RESULTS_DIR / 'logs'),
    'log_wandb': False,
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f'✅ Config saved → {CONFIG_PATH}')
print()
with open(CONFIG_PATH) as f:
    print(f.read())

✅ Config saved → /content/drive/MyDrive/MM_HSD/configs/text_audio_only.yml

run: text_audio_T4
batch_size: 8
epochs: 40
lr: 1.0e-05
dropout: 0.4
loss: CE
patience: 10
lambda_l1: 1.0e-05
lambda_l2: 0.0001
load_modalities:
- text
- audio
include_modalities:
- text
- audio
fusion: concat
audio_mode: wav2vec
input_size_text: 768
input_size_audio: 1024
input_size_ocr: 768
input_size_video: 768
hidden_size_1_general: 128
hidden_size_2_general: 128
hidden_size_2_audio: 80
output_size: 64
train_split_mode: automatic
cross_fold: true
folds: 5
dir_frames: null
dir_audio: /content/drive/MyDrive/MM_HSD/data/features/audio
dir_text: /content/drive/MyDrive/MM_HSD/data/features/text/embeddings_detoxify.json
dir_ocr: null
out_dir: /content/drive/MyDrive/MM_HSD/results/text_audio
log_dir: /content/drive/MyDrive/MM_HSD/results/text_audio/logs
log_wandb: false



In [ ]:
# ── Add repo to Python path so imports work ───────────
import sys
REPO_DIR  = DIRS['code'] / 'mm-hsd'
TRAIN_PY  = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'train.py'

sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'Repo    : {REPO_DIR}')
print(f'Train   : {TRAIN_PY}')
print(f'Config  : {CONFIG_PATH}')
print(f'Training script exists: {TRAIN_PY.exists()}')
print(f'Config exists         : {CONFIG_PATH.exists()}')

Repo    : /content/drive/MyDrive/MM_HSD/code/mm-hsd
Train   : /content/drive/MyDrive/MM_HSD/code/mm-hsd/src/mm_hsd/scripts/train.py
Config  : /content/drive/MyDrive/MM_HSD/configs/text_audio_only.yml
Training script exists: True
Config exists         : True


In [ ]:
# ── Add repo to Python path so imports work ───────────
import sys
import os

REPO_DIR  = DIRS['code'] / 'mm-hsd'
TRAIN_PY  = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'train.py'

# 1. Add to current notebook sys.path
sys.path.insert(0, str(REPO_DIR / 'src'))

# 2. Add to PYTHONPATH environment variable so subprocesses (like train.py) can find it!
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src') + ':' + os.environ.get('PYTHONPATH', '')

print(f'Repo    : {REPO_DIR}')
print(f'Train   : {TRAIN_PY}')
print(f'Config  : {CONFIG_PATH}')
print(f'Training script exists: {TRAIN_PY.exists()}')
print(f'Config exists         : {CONFIG_PATH.exists()}')


Repo    : /content/drive/MyDrive/MM_HSD/code/mm-hsd
Train   : /content/drive/MyDrive/MM_HSD/code/mm-hsd/src/mm_hsd/scripts/train.py
Config  : /content/drive/MyDrive/MM_HSD/configs/text_audio_only.yml
Training script exists: True
Config exists         : True


In [ ]:
!sed -i 's/argparse.Namespce/argparse.Namespace/g' {REPO_DIR}/src/mm_hsd/scripts/loader.py


In [ ]:
# ── Run training ──────────────────────────────────────
# Expected time on T4: ~20–40 minutes for 40 epochs × 5 folds
# Expected M-F1: ~0.70–0.75 (text+audio baseline from paper)

import subprocess, sys

cmd = [
    sys.executable,
    str(TRAIN_PY),
    '-c', str(CONFIG_PATH),
    '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
]

print('🚀 Launching training...')
print('Command:', ' '.join(cmd))
print('-' * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Stream output live
for line in process.stdout:
    print(line, end='')

process.wait()
print('-' * 60)
if process.returncode == 0:
    print('\n✅ Training complete!')
else:
    print(f'\n❌ Training failed with code {process.returncode}')

🚀 Launching training...
Command: /usr/bin/python3 /content/drive/MyDrive/MM_HSD/code/mm-hsd/src/mm_hsd/scripts/train.py -c /content/drive/MyDrive/MM_HSD/configs/text_audio_only.yml --device cuda
------------------------------------------------------------
device  cuda
Starting training for fold 1
Epoch: 0
Training       loss: 0.8847, F1-M: 0.4950, F1-H: 0.5378, P-H: 0.4268, R-H: 0.7266
Validation     loss: 0.8016, F1-M: 0.6118, F1-H: 0.6383, P-H: 0.5128, R-H: 0.8451
Epoch: 1
Training       loss: 0.8063, F1-M: 0.5405, F1-H: 0.5625, P-H: 0.4582, R-H: 0.7301
Validation     loss: 0.7580, F1-M: 0.6744, F1-H: 0.6772, P-H: 0.5751, R-H: 0.8310
Epoch: 2
Training       loss: 0.7835, F1-M: 0.5818, F1-H: 0.5907, P-H: 0.4934, R-H: 0.7399
Validation     loss: 0.7255, F1-M: 0.7078, F1-H: 0.6998, P-H: 0.6151, R-H: 0.8216
Epoch: 3
Training       loss: 0.7476, F1-M: 0.6086, F1-H: 0.6082, P-H: 0.5180, R-H: 0.7418
Validation     loss: 0.6956, F1-M: 0.7282, F1-H: 0.7132, P-H: 0.6447, R-H: 0.8099
Epoch: 4
T

In [ ]:
# ── Load and display results ──────────────────────────
import pandas as pd
import glob

RESULTS_DIR = DIRS['results'] / 'text_audio'

# Look for results CSV(s)
csv_files = list(RESULTS_DIR.rglob('**/*.csv'))
print(f'Found {len(csv_files)} results file(s):')

for csv_file in csv_files:
    print(f'\n📄 {csv_file.name}')
    df = pd.read_csv(csv_file)
    print(df.to_string())

print('\n📌 Expected baseline from paper:')
print('   Text only  : ~0.65–0.68 M-F1')
print('   Audio only : ~0.60–0.63 M-F1')
print('   Text+Audio : ~0.70–0.75 M-F1  ← Your target')
print('   Full model : 0.874 M-F1')

Found 5 results file(s):

📄 metric_logs_run_text_audio_T4_fold0.csv
    Unnamed: 0  loss_train  f1_train_macro  f1_train_hate  auc_roc_train  precision_train_macro  precision_train_hate  recall_train_macro  recall_train_hate  loss_val  f1_val_macro  f1_val_hate  auc_roc_val  precision_val_macro  precision_val_hate  recall_val_macro  recall_val_hate  accuracy_train  accuracy_val
0            0    0.884701        0.494986       0.537772       0.561332               0.540169              0.426829            0.536176           0.726644  0.801556      0.611832     0.638298     0.764588             0.663190            0.512821          0.651107         0.845070        0.498611      0.613636
1            1    0.806274        0.586108       0.587258       0.690773               0.610657              0.489607            0.610401           0.733564  0.758039      0.736972     0.716049     0.825755             0.742211            0.637363          0.751308         0.816901        0.586111      0.

In [ ]:
## 🎧 Cell 10 — Run Audio Only Model

import yaml, sys, os, torch
from pathlib import Path

# ── Patch loader.py argparse typo ──────
loader_py = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'loader.py'
loader_text = loader_py.read_text()
if 'argparse.Namespce' in loader_text:
    loader_py.write_text(loader_text.replace('argparse.Namespce', 'argparse.Namespace'))
    print('✅ Patched loader.py argparse typo')
# ── Patch CombinedModel.forward() unimodal bug in official repo ───
model_setup_py = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'model_setup.py'
ms_text = model_setup_py.read_text()
if 'self.unimodal]' in ms_text:
    ms_text = ms_text.replace(
        'modality_outputs[self.unimodal]',
        'modality_outputs[include_modalities[0]]'
    )
    model_setup_py.write_text(ms_text)
    print('✅ Patched model_setup.py unimodal bug')


# ── Ensure mm_hsd is importable ────────
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Write audio config ─────────────────
AUDIO_DIR = DIRS['results'] / 'audio_only'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

config_audio = config.copy()
config_audio['run'] = 'audio_only_T4'
config_audio['include_modalities'] = ['audio']
config_audio['load_modalities'] = ['audio']
config_audio['fusion'] = 'concat'
config_audio['out_dir'] = str(AUDIO_DIR)
config_audio['log_dir'] = str(AUDIO_DIR / 'logs')

CONFIG_AUDIO_PATH = DIRS['configs'] / 'audio_only.yml'
with open(CONFIG_AUDIO_PATH, 'w') as f:
    yaml.dump(config_audio, f, default_flow_style=False, sort_keys=False)

# ── Run training in-process ────────────
print('🚀 Launching Audio Only training...')

# Reload mm_hsd so patched files take effect
to_del = [k for k in sys.modules if k.startswith('mm_hsd')]
for k in to_del:
    del sys.modules[k]

from mm_hsd.scripts.train import get_argparser, main as train_main

sys.argv = ['train.py', '-c', str(CONFIG_AUDIO_PATH),
            '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
args = get_argparser().parse_args()
train_main(args)

print('✅ Audio Only training complete!')


✅ Patched model_setup.py unimodal bug
🚀 Launching Audio Only training...
device  cuda
Starting training for fold 1
Epoch: 0
Training       loss: 0.7749, F1-M: 0.5173, F1-H: 0.4121, P-H: 0.4234, R-H: 0.4014
Validation     loss: 0.7534, F1-M: 0.4849, F1-H: 0.2391, P-H: 0.5238, R-H: 0.1549
Epoch: 1
Training       loss: 0.7744, F1-M: 0.5069, F1-H: 0.3959, P-H: 0.4105, R-H: 0.3824
Validation     loss: 0.7454, F1-M: 0.4887, F1-H: 0.2418, P-H: 0.5514, R-H: 0.1549
Epoch: 2
Training       loss: 0.7697, F1-M: 0.5046, F1-H: 0.3942, P-H: 0.4085, R-H: 0.3809
Validation     loss: 0.7364, F1-M: 0.5201, F1-H: 0.3112, P-H: 0.5513, R-H: 0.2300
Epoch: 3
Training       loss: 0.7507, F1-M: 0.5137, F1-H: 0.4030, P-H: 0.4206, R-H: 0.3869
Validation     loss: 0.7296, F1-M: 0.5332, F1-H: 0.3352, P-H: 0.5621, R-H: 0.2500
Epoch: 4
Training       loss: 0.7460, F1-M: 0.5185, F1-H: 0.4110, P-H: 0.4256, R-H: 0.3977
Validation     loss: 0.7286, F1-M: 0.5291, F1-H: 0.3265, P-H: 0.5617, R-H: 0.2394
Epoch: 5
Training   

In [ ]:
## 📝 Cell 11 — Run Text Only Model

import yaml, sys, os, torch
from pathlib import Path

# ── Patch loader.py argparse typo ──────
loader_py = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'loader.py'
loader_text = loader_py.read_text()
if 'argparse.Namespce' in loader_text:
    loader_py.write_text(loader_text.replace('argparse.Namespce', 'argparse.Namespace'))
    print('✅ Patched loader.py argparse typo')
# ── Patch CombinedModel.forward() unimodal bug in official repo ───
model_setup_py = REPO_DIR / 'src' / 'mm_hsd' / 'scripts' / 'model_setup.py'
ms_text = model_setup_py.read_text()
if 'self.unimodal]' in ms_text:
    ms_text = ms_text.replace(
        'modality_outputs[self.unimodal]',
        'modality_outputs[include_modalities[0]]'
    )
    model_setup_py.write_text(ms_text)
    print('✅ Patched model_setup.py unimodal bug')


# ── Ensure mm_hsd is importable ────────
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Write text config ──────────────────
TEXT_DIR = DIRS['results'] / 'text_only'
TEXT_DIR.mkdir(parents=True, exist_ok=True)

config_text = config.copy()
config_text['run'] = 'text_only_T4'
config_text['include_modalities'] = ['text']
config_text['load_modalities'] = ['text']
config_text['fusion'] = 'concat'
config_text['out_dir'] = str(TEXT_DIR)
config_text['log_dir'] = str(TEXT_DIR / 'logs')

CONFIG_TEXT_PATH = DIRS['configs'] / 'text_only.yml'
with open(CONFIG_TEXT_PATH, 'w') as f:
    yaml.dump(config_text, f, default_flow_style=False, sort_keys=False)

# ── Run training in-process ────────────
print('🚀 Launching Text Only training...')

# Reload mm_hsd so patched files take effect
to_del = [k for k in sys.modules if k.startswith('mm_hsd')]
for k in to_del:
    del sys.modules[k]

from mm_hsd.scripts.train import get_argparser, main as train_main

sys.argv = ['train.py', '-c', str(CONFIG_TEXT_PATH),
            '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
args = get_argparser().parse_args()
train_main(args)

print('✅ Text Only training complete!')


🚀 Launching Text Only training...
device  cuda
Starting training for fold 1
Epoch: 0
Training       loss: 0.8633, F1-M: 0.5109, F1-H: 0.3282, P-H: 0.4568, R-H: 0.2561
Validation     loss: 0.7393, F1-M: 0.5076, F1-H: 0.2558, P-H: 0.7333, R-H: 0.1549
Epoch: 1
Training       loss: 0.7810, F1-M: 0.5636, F1-H: 0.4131, P-H: 0.5250, R-H: 0.3426
Validation     loss: 0.6817, F1-M: 0.6453, F1-H: 0.4879, P-H: 0.7833, R-H: 0.3944
Epoch: 2
Training       loss: 0.7358, F1-M: 0.5963, F1-H: 0.4640, P-H: 0.5663, R-H: 0.3956
Validation     loss: 0.6415, F1-M: 0.6938, F1-H: 0.5734, P-H: 0.7798, R-H: 0.5023
Epoch: 3
Training       loss: 0.7111, F1-M: 0.6158, F1-H: 0.4978, P-H: 0.5813, R-H: 0.4403
Validation     loss: 0.6135, F1-M: 0.7245, F1-H: 0.6251, P-H: 0.7813, R-H: 0.5704
Epoch: 4
Training       loss: 0.7063, F1-M: 0.6275, F1-H: 0.5181, P-H: 0.5899, R-H: 0.4675
Validation     loss: 0.5930, F1-M: 0.7442, F1-H: 0.6578, P-H: 0.7828, R-H: 0.6141
Epoch: 5
Training       loss: 0.6724, F1-M: 0.6429, F1-H: 0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch: 0
Training       loss: 0.8997, F1-M: 0.4531, F1-H: 0.1979, P-H: 0.3958, R-H: 0.1319
Validation     loss: 0.7923, F1-M: 0.3737, F1-H: 0.0000, P-H: 0.0000, R-H: 0.0000
Epoch: 1
Training       loss: 0.8053, F1-M: 0.4966, F1-H: 0.2677, P-H: 0.4936, R-H: 0.1840
Validation     loss: 0.7218, F1-M: 0.4981, F1-H: 0.2330, P-H: 0.3750, R-H: 0.1690
Epoch: 2
Training       loss: 0.7316, F1-M: 0.5461, F1-H: 0.3512, P-H: 0.5576, R-H: 0.2616
Validation     loss: 0.6743, F1-M: 0.5788, F1-H: 0.3809, P-H: 0.4919, R-H: 0.3239
Epoch: 3
Training       loss: 0.7002, F1-M: 0.5782, F1-H: 0.4076, P-H: 0.5884, R-H: 0.3212
Validation     loss: 0.6457, F1-M: 0.6224, F1-H: 0.4628, P-H: 0.5436, R-H: 0.4225
Epoch: 4
Training       loss: 0.6673, F1-M: 0.6074, F1-H: 0.4572, P-H: 0.6112, R-H: 0.3799
Validation     loss: 0.6303, F1-M: 0.6488, F1-H: 0.5127, P-H: 0.5736, R-H: 0.4845
Epoch: 5
Training       loss: 0.6461, F1-M: 0.6317, F1-H: 0.4964, P-H: 0.6332, R-H: 0.4246
Validation     loss: 0.6221, F1-M: 0.6654, F